In [1]:
!pip install transformers datasets wandb scikit-learn -q

In [2]:
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast

genres = [
    "Children",
    "Comics, Graphic",
    "Fantasy, Paranormal",
    "Fiction",
    "History, Historical Fiction, Biography",
    "Mystery, Thriller & Crime",
    "Non-Fiction",
    "Poetry",
    "Romance"
]

label2id = {label: i for i, label in enumerate(genres)}
id2label = {i: label for i, label in enumerate(genres)}

print("✅ Labels defined:", len(id2label))
print(id2label)

✅ Labels defined: 9
{0: 'Children', 1: 'Comics, Graphic', 2: 'Fantasy, Paranormal', 3: 'Fiction', 4: 'History, Historical Fiction, Biography', 5: 'Mystery, Thriller & Crime', 6: 'Non-Fiction', 7: 'Poetry', 8: 'Romance'}


In [3]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
import os

print("📥 Loading dataset...")

try:
    
    df = pd.read_csv('/kaggle/input/goodreads-book-reviews1/goodreads_reviews.csv')
except:
    
    print("Using fallback sample data...")
    data = {
        'review': [
            "This fantasy book was amazing with dragons and magic!", 
            "A thrilling mystery that kept me guessing till the end.",
            "Great romance novel with beautiful love story.",
            "Interesting non-fiction about history and politics.",
        ] * 100,  # repeat to have enough samples
        'genre': ["Fantasy, Paranormal"]*100 + ["Mystery, Thriller & Crime"]*100 + 
                 ["Romance"]*100 + ["History, Historical Fiction, Biography"]*100
    }
    df = pd.DataFrame(data)

print("✅ Dataset loaded!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nGenre counts:\n", df['genre'].value_counts())

df = df.groupby('genre').sample(n=min(500, df['genre'].value_counts().min()), random_state=42)

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['genre'], random_state=42)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print("✅ Train/Test datasets created!")

📥 Loading dataset...
Using fallback sample data...
✅ Dataset loaded!
Shape: (400, 2)
Columns: ['review', 'genre']

Genre counts:
 genre
Fantasy, Paranormal                       100
Mystery, Thriller & Crime                 100
Romance                                   100
History, Historical Fiction, Biography    100
Name: count, dtype: int64
✅ Train/Test datasets created!


In [4]:
from transformers import DistilBertTokenizerFast

model_name = 'distilbert-base-uncased'
max_length = 512
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

def tokenize_function(examples):
    # Make sure the text column name is correct
    return tokenizer(
        examples['review'],      
        padding="max_length", 
        truncation=True, 
        max_length=max_length
    )

print("Tokenizing datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

def add_labels(example):
    example['labels'] = label2id[example['genre']]
    return example

train_dataset = train_dataset.map(add_labels)
test_dataset = test_dataset.map(add_labels)

# Set PyTorch format
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("✅ Tokenization completed!")
print("Final train size:", len(train_dataset))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing datasets...


Map:   0%|          | 0/320 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/320 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

✅ Tokenization completed!
Final train size: 320


In [5]:
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model_name = 'distilbert-base-uncased'
max_length = 512

tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
).to(device)

print("✅ Model loaded successfully!")
print(f"Number of labels: {model.config.num_labels}")

Using device: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded successfully!
Number of labels: 9


In [6]:
from kaggle_secrets import UserSecretsClient
import os
import wandb
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

secrets = UserSecretsClient()

os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')   # Optional but good to have

print("✅ Secrets loaded successfully!")

wandb.login(key=os.environ['WANDB_API_KEY'])
wandb.init(
    project="mlops-assignment2",
    name="distilbert-goodreads-run-1",
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 3e-5,
        "max_length": max_length,
        "dataset": "UCSD Goodreads (sample)",
        "platform": "Kaggle GPU T4",
    }
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name="distilbert-goodreads-run-1",
    fp16=True,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("🚀 Starting training with W&B tracking...")
trainer.train()

wandb.finish()
print("🎉 Training Completed with W&B!")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


✅ Secrets loaded successfully!


TypeError: the JSON object must be str, bytes or bytearray, not NoneType

In [ ]:
import json
from sklearn.metrics import classification_report

print("Running final evaluation...")
eval_results = trainer.evaluate()

print("\n=== FINAL RESULTS ===")
print(f"Eval Loss:     {eval_results['eval_loss']:.4f}")
print(f"Accuracy:      {eval_results.get('eval_accuracy', 0):.4f}")
print(f"F1 Score:      {eval_results.get('eval_f1', 0):.4f}")

predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)
labels = [item["labels"].item() for item in test_dataset]

unique_labels = sorted(set(labels))
target_names = [id2label[i] for i in unique_labels]

report = classification_report(
    labels, 
    preds, 
    labels=unique_labels,
    target_names=target_names,
    output_dict=True
)

with open("eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("\n✅ Evaluation report saved successfully!")
print("Number of classes in test set:", len(unique_labels))

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')

login(token=HF_TOKEN)

username = "BhavikSolanki"
repo_name = f"{username}/ML-OPS-Assignment2"

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print(f"\n🎉 SUCCESS! Model pushed to:")
print(f"https://huggingface.co/{repo_name}")